# Gold Layer — Joined Analytical Fact Table

In [0]:
from pyspark.sql import functions as F

df_prices = spark.table("silver_prices")
df_fg     = spark.table("silver_fear_greed")

print(f"Prices rows: {df_prices.count()}")
print(f"Fear & Greed rows: {df_fg.count()}")

In [0]:
df_prices_keyed = df_prices.withColumn(
    "ingestion_hour",
    F.date_trunc("hour", F.col("last_updated"))
)

df_fg_keyed = df_fg.withColumn(
    "ingestion_hour",
    F.date_trunc("hour", F.col("ingested_at"))
).select("ingestion_hour", "fear_greed_value", "value_classification")

In [0]:
df_joined = df_prices_keyed.join(df_fg_keyed, on="ingestion_hour", how="left")
print(f"Joined rows: {df_joined.count()}")

In [0]:
df_gold = df_joined.select(
    F.col("ingestion_hour"),
    F.col("last_updated"),
    F.col("id").alias("coin_id"),
    F.col("symbol"),
    F.col("name"),
    F.col("current_price"),
    F.col("market_cap"),
    F.col("market_cap_rank"),
    F.col("total_volume"),
    F.col("high_24h"),
    F.col("low_24h"),
    F.col("price_change_24h"),
    F.col("price_change_percentage_24h"),
    F.col("circulating_supply"),
    F.col("fear_greed_value"),
    F.col("value_classification"),
    F.when(F.abs(F.col("price_change_percentage_24h")) >= 5, "High")
     .when(F.abs(F.col("price_change_percentage_24h")) >= 2, "Medium")
     .otherwise("Low")
     .alias("volatility_category")
)

print(f"Gold rows: {df_gold.count()}")
display(df_gold)

In [0]:
df_gold.write.format("delta").mode("overwrite").saveAsTable("gold_price_sentiment")
print("Gold table written: gold_price_sentiment")

In [0]:
display(spark.sql("SHOW TABLES"))

In [0]:
display(spark.sql("""
    SELECT
        value_classification,
        fear_greed_value,
        COUNT(*)                                    AS coin_readings,
        ROUND(AVG(price_change_percentage_24h), 4)  AS avg_price_change_pct,
        ROUND(AVG(current_price), 2)                AS avg_price,
        ROUND(AVG(total_volume), 0)                 AS avg_volume
    FROM gold_price_sentiment
    WHERE fear_greed_value IS NOT NULL
    GROUP BY value_classification, fear_greed_value
    ORDER BY fear_greed_value ASC
"""))